In [ ]:
from langgraph.graph import StateGraph , START , END
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from langchain_google_genai import GoogleGenerativeAI , ChatGoogleGenerativeAI

from dotenv import load_dotenv
load_dotenv()
from typing import TypedDict , Annotated
from pydantic import BaseModel , Field
import operator

In [27]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-32B",
    task="text-generation",
    provider="auto",
    temperature=0.7,
    max_new_tokens=100000
)

model = ChatHuggingFace(llm = llm)

In [28]:
essay = """
The AI revolution is transforming the way people live, work, and communicate. Artificial Intelligence can perform tasks such as understanding language, analyzing data, recognizing images, and assisting in decision-making. It is being used in education, healthcare, transportation, business, and entertainment. AI helps automate repetitive tasks, improve productivity, and provide faster solutions to complex problems. However, it also creates challenges, including job displacement, privacy concerns, misinformation, and the need for responsible development. As AI continues to advance, humans must learn to use it wisely and ethically. The AI revolution has enormous potential to improve society while changing the future of work and technology.
"""

In [29]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [ ]:
class UPSCState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_score:Annotated[list[int],operator.add]
    avg_score:float

In [ ]:
def evaluate_language(state: UPSCState):

    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def evaluate_analysis(state: UPSCState):

    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def evaluate_thought(state: UPSCState):

    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def final_evaluation(state: UPSCState):

    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [ ]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [ ]:
intial_state = {
    'essay': essay
}

workflow.invoke(intial_state)